# DiffPIR — Google Colab demo

Plug-and-play image restoration via diffusion. Runs SR x4 / Gaussian deblur / box inpaint on FFHQ checkpoint.

**Runtime → Change runtime type → GPU (T4)** before executing. CPU works but is ~50x slower.

## 1. Clone repo + motionblur

In [ ]:
import os
if not os.path.isdir('DiffPIR'):
    !git clone https://github.com/sfoucher/DiffPIR
%cd DiffPIR
if not os.path.isdir('motionblur'):
    !git clone https://github.com/LeviBorodenko/motionblur

## 2. Install deps

Colab ships torch + opencv + scipy + matplotlib. Add the rest.

In [ ]:
!pip install --quiet hdf5storage==0.1.19 blobfile==2.0.1 lpips==0.1.4 gdown

## 3. Download FFHQ checkpoint (~2 GB)

In [ ]:
import os, gdown
os.makedirs('model_zoo', exist_ok=True)
ckpt = 'model_zoo/diffusion_ffhq_10m.pt'
if not os.path.exists(ckpt):
    gdown.download(id='1BGwhRWUoguF-D8wlZ65tf227gp3cDUDh', output=ckpt, quiet=False)
print('checkpoint:', ckpt, os.path.getsize(ckpt))

## 4. GPU sanity check

In [ ]:
import torch
print('cuda:', torch.cuda.is_available(), '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only')

## 5. Pick task + tweak config

Switch `TASK` between `'sr'`, `'deblur'`, `'inpaint'`. GPU → keep `iter_num=100` for paper-quality. CPU → drop to 20.

In [ ]:
import yaml, shutil
TASK = 'sr'   # 'sr' | 'deblur' | 'inpaint'
ITER_NUM = 100 if torch.cuda.is_available() else 20

src_cfg = {'sr': 'configs/sisr.yaml', 'deblur': 'configs/deblur.yaml', 'inpaint': 'configs/inpaint.yaml'}[TASK]
with open(src_cfg) as f:
    cfg = yaml.safe_load(f)
cfg['testset_name'] = 'demo_test'
cfg['iter_num'] = ITER_NUM
cfg['batch_size'] = 1 if not torch.cuda.is_available() else cfg.get('batch_size', 1)
cfg['calc_LPIPS'] = False
cfg['save_L'] = True
cfg['save_E'] = True
cfg_path = f'configs/{TASK}_colab.yaml'
with open(cfg_path, 'w') as f:
    yaml.dump(cfg, f)
print(cfg_path, '→', cfg)

## 6. Run

Streams stderr live (logger writes there).

In [ ]:
!python main_ddpir.py --opt {cfg_path}

## 7. Display results

In [ ]:
import glob, os, cv2, matplotlib.pyplot as plt
out_dirs = sorted(glob.glob(f'results/demo_test_{TASK}_*'), key=os.path.getmtime)
assert out_dirs, 'no result dir'
out = out_dirs[-1]
print('result dir:', out)
imgs = sorted(f for f in os.listdir(out) if f.lower().endswith(('.png', '.jpg')))
print(imgs)
n = len(imgs)
fig, axes = plt.subplots(1, n, figsize=(4*n, 4))
if n == 1:
    axes = [axes]
for ax, f in zip(axes, imgs):
    img = cv2.cvtColor(cv2.imread(os.path.join(out, f)), cv2.COLOR_BGR2RGB)
    ax.imshow(img); ax.set_title(f, fontsize=8); ax.axis('off')
plt.tight_layout(); plt.show()